# 🔍 Project Inspector — stressTest v2
**Структура движка, БД и данных для финансового аналитика**


In [ ]:
COMPANY_ID = 'rusal'

import sys, logging, warnings, sqlite3
warnings.filterwarnings('ignore'); logging.disable(logging.CRITICAL)
from pathlib import Path
for p in Path('.').absolute().parents:
    if (p / 'data_mart_v2.db').exists():
        sys.path.insert(0, str(p)); break
from engine import DB_PATH
from engine.database.repository import Repository
from engine.orchestrator import build_model
try:
    from IPython.display import display, HTML
except ImportError:
    display = lambda *a, **kw: None
    class HTML:
        def __init__(self, *a): pass
import pandas as pd

try:
    r = build_model(COMPANY_ID, run_preprocessor=False, run_model=True)
    bs = max(r.model_result.bs_diffs.values())
    model_ok = True
    print(f'Model OK: BS={bs:.0f} | Company: {COMPANY_ID.upper()}')
except Exception as e:
    model_ok = False
    print(f'Model ERROR: {e}')


---
## 1 · Схема базы данных


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print(f'Tables: {len(tables)}')
print(f'{"Table":<40} {"Rows":>8} {"Cols":>5}')
print('-'*55)
for (tbl,) in tables:
    cnt = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    cols = len(conn.execute(f'PRAGMA table_info({tbl})').fetchall())
    if cnt > 0:
        print(f'{tbl:<40} {cnt:>8,} {cols:>5}')
conn.close()


---
## 2 · Покрытие данных IS/BS/CF


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
YEARS = list(range(2011, 2026))
for stmt, label in [('history_is','IS'), ('history_bs','BS'), ('history_cf','CF')]:
    metrics = conn.execute(f'''
        SELECT DISTINCT h.metric FROM {stmt} h
        JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? ORDER BY h.metric
    ''', (COMPANY_ID,)).fetchall()
    print(f'\n{label}: {len(metrics)} metrics')
    for (m,) in metrics:
        yrs = conn.execute(f'''
            SELECT p.year FROM {stmt} h JOIN periods p ON h.period_id=p.period_id
            WHERE h.company_id=? AND h.metric=? ORDER BY p.year
        ''', (COMPANY_ID, m)).fetchall()
        yr_list = [r[0] for r in yrs]
        filled = len(yr_list)
        status = 'OK' if filled >= 10 else ('FEW' if filled >= 5 else 'LOW')
        print(f'  {m:<40} {yr_list[0]}-{yr_list[-1]} ({filled}yr) [{status}]')
conn.close()


---
## 3 · Препроцессор — параметры модели


In [ ]:
with Repository(DB_PATH) as repo:
    groups = repo.query('''
        SELECT metric_group, COUNT(*) n FROM preprocess_metrics
        WHERE company_id=? GROUP BY metric_group ORDER BY metric_group
    ''', (COMPANY_ID,))
    print(f'Preprocess groups: {len(groups)}')
    print(f'{"Group":<25} {"Metrics":>8} Recommended values')
    print('-'*70)
    for g in groups:
        recs = repo.query('''
            SELECT metric_name, value FROM preprocess_metrics
            WHERE company_id=? AND metric_group=? AND metric_name LIKE "%recommended%" AND year=-1
            LIMIT 3
        ''', (COMPANY_ID, g['metric_group']))
        rec_str = ', '.join(f'{r["metric_name"].replace("_recommended","")}={r["value"]:.4f}' for r in recs if r['value'])
        print(f'{g["metric_group"]:<25} {g["n"]:>8} {rec_str[:50]}')


---
## 4 · Долговые инструменты


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
di = conn.execute('''
    SELECT instrument_name, opening_balance, interest_rate, rate_type, maturity_date, currency
    FROM debt_instruments WHERE company_id=? ORDER BY opening_balance DESC
''', (COMPANY_ID,)).fetchall()
total = sum(r[1] or 0 for r in di)
print(f'Debt instruments: {len(di)}, total ${total/1e9:.2f}B')
print(f'{"Instrument":<35} {"$M":>8} {"Rate":>7} {"Type":>8} {"Mat":>10} {"CCY":>4}')
print('-'*75)
for r in di[:15]:
    print(f'{str(r[0])[:35]:<35} {(r[1] or 0)/1e6:>8,.0f} {(r[2] or 0)*100:>6.1f}% {str(r[3] or "fixed"):>8} {str(r[4] or "?")[:10]:>10} {str(r[5] or "USD"):>4}')
if len(di) > 15:
    print(f'... +{len(di)-15} more')
print(f'{"TOTAL":<35} {total/1e6:>8,.0f}')
conn.close()


---
## 5 · Статус модели и ковенанты


In [ ]:
from engine.rating.core import RatingEngine, RatingConfig, CreditMetrics
eng = RatingEngine(RatingConfig(methodology='sp'))

r2 = build_model(COMPANY_ID, run_preprocessor=False, run_model=True)
mr = r2.model_result

print(f'{"Yr":<6}{"Rev$B":>7}{"EBITDA%":>9}{"NI$M":>7}{"ND/EB":>7}{"Rating":>8}')
for yr, s in sorted(mr.years.items()):
    nd = abs(s.long_term_debt or 0)+abs(s.short_term_debt or 0)-(s.cash or 0)
    eb = s.ebitda or 1
    try:
        m = CreditMetrics.from_year_state(s, yr)
        rt = eng.calculate(m)['rating']
    except: rt = '?'
    print(f'{yr:<6}{s.revenue/1e9:>7.2f}{s.ebitda/s.revenue*100:>8.1f}%{s.net_income/1e6:>7.0f}{nd/eb:>6.2f}x  {rt}')


---
## 7 · Backtesting — точность модели
Автоматический прогон: train на N-1 лет, test на последние 1-2 года.


In [ ]:
import yaml, copy
from engine.model.loader import ModelInputLoader
from engine.model.core import ThreeStatementModel

# Determine last history year and backtest windows
conn = sqlite3.connect(str(DB_PATH))
max_yr = conn.execute('''
    SELECT MAX(p.year) FROM history_is h JOIN periods p ON h.period_id=p.period_id
    WHERE h.company_id=?
''', (COMPANY_ID,)).fetchone()[0]
conn.close()

WINDOWS = [
    {'name': '1-year ahead', 'train_end': max_yr - 2, 'test': [max_yr - 1, max_yr]},
    {'name': '2-year ahead', 'train_end': max_yr - 3, 'test': [max_yr - 2, max_yr - 1]},
]

cfg_path = Path(f'companies/{COMPANY_ID}/configs/project.yaml')
with open(cfg_path) as f:
    base_cfg = yaml.safe_load(f)

# Load actuals
actuals = {}
with Repository(DB_PATH) as repo:
    for yr in range(max_yr - 3, max_yr + 1):
        actuals[yr] = repo.get_history_year(COMPANY_ID, 'IS', yr)

METRICS = ['revenue', 'cogs', 'ebitda', 'net_income']

print(f'Backtesting {COMPANY_ID.upper()}: history ends {max_yr}')
print(f'{"="*70}')

all_results = {}

for window in WINDOWS:
    train_end = window['train_end']
    test_years = window['test']
    
    cfg_bt = copy.deepcopy(base_cfg)
    cfg_bt['model']['standard']['periods']['history_end_year'] = train_end
    cfg_bt['model']['standard']['periods']['forecast_start_year'] = train_end + 1
    cfg_bt['model']['standard']['periods']['forecast_end_year'] = max(test_years)
    cfg_bt['model']['standard']['periods']['forecast_years'] = len(test_years)
    # Disable complex modes for clean backtest
    custom = cfg_bt.get('model', {}).get('custom', {})
    if custom.get('revenue', {}).get('segment_modeling'):
        custom['revenue']['segment_modeling'] = False
    if custom.get('cogs', {}).get('mode') == 'component':
        custom.pop('cogs', None)
    
    bt_path = Path(f'companies/{COMPANY_ID}/configs/_bt_temp.yaml')
    with open(bt_path, 'w') as f:
        yaml.dump(cfg_bt, f, allow_unicode=True, sort_keys=False)
    
    try:
        with Repository(DB_PATH) as repo:
            loader = ModelInputLoader(COMPANY_ID, repo, bt_path, 'base')
            hist, mcfg = loader.load()
            result = ThreeStatementModel(hist, mcfg).run()
        
        print(f'\n--- {window["name"]}: train ≤{train_end}, test {test_years} ---')
        print(f'{"Metric":<15} {"Year":<5} {"Forecast":>10} {"Actual":>10} {"Error":>7}')
        
        for m in METRICS:
            for yr in test_years:
                s = result.years.get(yr)
                if not s: continue
                fc = getattr(s, m, 0) or 0
                ac = actuals.get(yr, {}).get(m, 0) or 0
                if m == 'cogs': fc = abs(fc); ac = abs(ac)
                if ac == 0: continue
                err = (fc - ac) / abs(ac) * 100
                status = 'OK' if abs(err) < 15 else ('~' if abs(err) < 30 else '!!')
                all_results.setdefault(m, {}).setdefault(window['name'], []).append(abs(err))
                print(f'{m:<15} {yr:<5} ${fc/1e6:>8,.0f}M ${ac/1e6:>8,.0f}M {err:>+6.1f}% {status}')
    except Exception as e:
        print(f'  Error: {e}')
    finally:
        bt_path.unlink(missing_ok=True)

# MAPE summary table
print(f'\n{"="*70}')
print(f'MAPE SUMMARY:')
print(f'{"Metric":<15}', end='')
for w in WINDOWS:
    print(f'{w["name"]:>15}', end='')
print(f'{"Grade":>8}')
print('-'*50)

for m in METRICS:
    print(f'{m:<15}', end='')
    worst = 0
    for w in WINDOWS:
        errs = all_results.get(m, {}).get(w['name'], [])
        mape = sum(errs) / len(errs) if errs else 999
        worst = max(worst, mape)
        print(f'{mape:>14.1f}%', end='')
    grade = 'A' if worst < 10 else ('B' if worst < 20 else ('C' if worst < 35 else 'F'))
    print(f'{grade:>8}')

print(f'\nInterpretation:')
print(f'  A (<10%):  Production-ready for credit analysis')
print(f'  B (<20%):  Good for scenario planning')
print(f'  C (<35%):  Directional only')
print(f'  F (>35%):  Volatile item — use stress scenarios instead')


---
## 6 · Диагностика


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
issues = []

# Check IS critical metrics
for m in ['revenue','cogs','ebit','net_income','interest_expense']:
    cnt = conn.execute('''
        SELECT COUNT(*) FROM history_is h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND h.metric=?
    ''', (COMPANY_ID, m)).fetchone()[0]
    if cnt == 0:
        issues.append(f'CRITICAL: IS metric "{m}" missing')
    elif cnt < 5:
        issues.append(f'WARNING: IS metric "{m}" only {cnt} years')

# Check BS critical
for m in ['cash','ppe_net','long_term_debt','total_equity','total_assets']:
    cnt = conn.execute('''
        SELECT COUNT(*) FROM history_bs h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND h.metric=?
    ''', (COMPANY_ID, m)).fetchone()[0]
    if cnt == 0:
        issues.append(f'CRITICAL: BS metric "{m}" missing')

# Check debt
di_cnt = conn.execute('SELECT COUNT(*) FROM debt_instruments WHERE company_id=?', (COMPANY_ID,)).fetchone()[0]
if di_cnt == 0:
    issues.append('CRITICAL: No debt instruments')

# Check preprocess
pp_cnt = conn.execute('SELECT COUNT(*) FROM preprocess_metrics WHERE company_id=?', (COMPANY_ID,)).fetchone()[0]
if pp_cnt < 100:
    issues.append(f'WARNING: Only {pp_cnt} preprocess metrics (expected >500)')

conn.close()

if issues:
    print(f'Found {len(issues)} issues:')
    for i in issues:
        print(f'  {i}')
else:
    print('All checks passed — data ready for modeling')
